# Preprocess Reaching Session

Interactive notebook for preprocessing SpikeGLX recording sessions. Uses widget-based workflows for session initialization, file alignment, antidromic stimulation analysis, video alignment, and behavioral state scoring.

**Requirements:** Place this notebook in the session directory containing your recording data. A `session_config.json` file will be created if not present.

In [1]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path

# Interactive widgets
import ipywidgets as widgets
from IPython.display import display, clear_output

# Import the modernized MIMO session framework
from mimo_pack.core.session import Session
from mimo_pack.util.scaffold import create_session_config

# Core preprocessing modules  
from mimo_pack.preprocess.session import preprocess_session
from mimo_pack.preprocess.antidromic import get_antistim_times, create_stimulation_summary
from mimo_pack.preprocess.video import tif_to_video, create_video_rois, process_and_align_video

# Analysis modules
from mimo_pack.analysis.evoked import evoked_field_summary
from mimo_pack.analysis.beh_state import states_lfp_only, plot_band_power_states

# Plotting modules
from mimo_pack.plot.antidromic import plot_antidromic_summary
from mimo_pack.plot.map import wave_map_xr

print("✅ MIMO imports successful - using modernized session-driven workflow!")

✅ MIMO imports successful - using modernized session-driven workflow!


## 0. Session Initialization

**Instructions:**
1. Verify the **Session Dir** path points to your recording session folder
2. Click **🔍 Check Directory** to verify the directory structure and find data files
3. Enable **Create default config if missing** if no `session_config.json` exists
4. Click **🚀 Load Session** to initialize the session object for all subsequent widgets

In [2]:
# Import and display SessionInitializer widget
from mimo_pack.widgets import SessionInitializer

# Global variable to store the session
session = None

# Create and display the interactive session initializer
initializer = SessionInitializer()
initializer.display()

# After loading, access session via: session = initializer.session

## 1. Preprocess and Align SpikeGLX Files

**Instructions:**
1. Configure the **AP Channel** and **NIDQ Channel** sync settings
2. Click **⚡ Run Preprocessing** to align all SpikeGLX data streams
3. Use **💾 Save Config** to store settings for reproducibility

In [3]:
# Import and display SpikeGLX Preprocessor widget
from mimo_pack.widgets import SpikeGLXPreprocessor

# Get session from initializer
session = initializer.session

if session is not None:
    preprocessor = SpikeGLXPreprocessor(session)
    preprocessor.display()
else:
    print("❌ Session not found. Please run the session initialization cell first.")
    print("💡 The session should be loaded before running preprocessing.")

## 2. Antidromic Stimulation Analysis

**Instructions:**
1. Set the **Stim Channel** and click **🔍 Extract Data**
2. Review the statistics table showing timing, pulse count, and frequency
3. Select bad groups in the **Bad Groups** list, then annotate valid groups with region/current
4. Click **🔧 Apply Configuration** then **💾 Save to Session** to store your choices
5. Use **📊 Generate QC Reports** to create analysis plots for all probes

In [4]:
# Import and display Antidromic Analysis widget
from mimo_pack.widgets import AntidromicAnalysisWidget

# Get session from initializer
session = initializer.session

if session is not None:
    analysis_widget = AntidromicAnalysisWidget(session)
    analysis_widget.display()
else:
    print("❌ Session not found. Please run the session initialization cell first.")
    print("💡 The session should be loaded before running antidromic analysis.")

## 3. Video Processing and Alignment

**Instructions:**
1. Set the **Image Dir** path and click **🎬 Convert Images to Video**, or enter a **Video Path** and click **📹 Load Video**
2. Use the **Frame** slider and **Brightness** slider to navigate to a clear frame showing the barcode LEDs
3. Click and drag on the image to draw 8 rectangles around each barcode LED
4. Fine-tune coordinates in the ROI table if needed, click **🔄 Update Display** to see changes
5. Click **🎯 Process & Align Video** to synchronize video with neural timestamps
6. Use **💾 Save Config** to store ROI coordinates for reprocessing

In [ ]:
# Import and display Video Alignment widget
%matplotlib widget
from mimo_pack.widgets import VideoAlignmentWidget

# Get session from initializer
session = initializer.session

if session is not None:
    video_widget = VideoAlignmentWidget(session)
    video_widget.display()
    
    # Export variables for compatibility with downstream cells
    vid_path = video_widget.vid_path
    tbl_path = video_widget.tbl_path
    rois = video_widget.rois
else:
    print("❌ Session not found. Please run the session initialization cell first.")
    print("💡 The session should be loaded before running video preprocessing.")

## 4. Behavioral State Analysis

**Instructions:**
1. Set **Probe Index** and **LFP Channel**, then click **Load LFP** to load the data
2. Use **Visualize Channels** to view LFP across channels and identify a good channel
3. Adjust frequency band parameters (Delta, Theta, High, Noise) as needed
4. Click **Run Analysis** to compute behavioral states and generate QC plots
5. Use **Save Config** to store parameters for reproducibility

In [6]:
# Import and display Behavioral State Analysis widget
from mimo_pack.widgets import BehavioralStateWidget

# Get session from initializer
session = initializer.session

if session is not None:
    state_widget = BehavioralStateWidget(session)
    state_widget.display()
else:
    print("❌ Session not found. Please run the session initialization cell first.")
    print("💡 The session should be loaded before running behavioral state analysis.")

## Summary

**Generated Outputs:**
- SpikeGLX dclut files (aligned timestamps for all data streams)
- Antidromic analysis (QC reports and stimulation timing)
- Video alignment (synchronized behavioral video)
- Behavioral states (LFP-derived state classifications)

All parameters are saved in `session_config.json` for reproducibility.